# Data Preprocessing v2
**Perbaikan dari versi si X:**
- Turning Point otomatis menggunakan CUSUM (tidak hardcode)
- Ekstraksi fitur statistik time-domain dan frequency-domain per window
- Menghasilkan 2 dataset:
  - `raw` → persis seperti X: time, X, Y, Z (rata-rata per detik, tanpa ekstraksi statistik)
  - `stat_fft` → fitur statistik time-domain + spectral (tambahan baru)

## Konfigurasi Global

In [ ]:
# ─── Path ───────────────────────────────────────────────────────────────────
ROOT_PROJECT_DATASET = "/mnt/e/Mas Erge-Bearing_RUL/Datasets"  # WSL path
# ROOT_PROJECT_DATASET = r"E:\Mas Erge-Bearing_RUL\Datasets"   # Windows path (jika tidak pakai WSL)

BEARING_VARIATION_FOLDER = ("bearing_1",)
MAIN_BEARING = BEARING_VARIATION_FOLDER[0]

# ─── Sensor ─────────────────────────────────────────────────────────────────
SAMPLING_RATE = 2560  # Hz → 1 window = 2560 sampel = 1 detik

# ─── Smoothing & Downsampling (sama seperti versi X) ────────────────────────
SMOOTHING_WINDOW      = 3600 * 4   # 4 jam
DOWN_SAMPLING_FACTOR  = 10 * 60    # 1 titik per 10 menit
TESTING_INITIAL_POINT = 900        # offset awal dataset test

# ─── CUSUM Parameters ────────────────────────────────────────────────────────
CUSUM_BASELINE_RATIO  = 0.10  # 10% data awal dianggap kondisi sehat
CUSUM_K_FACTOR        = 0.5   # drift sensitivity
CUSUM_H_FACTOR        = 5.0   # decision threshold (false-alarm control)

In [ ]:
PLOT_STYLES = {
    "HI":      {"color": "black",  "linestyle": "-"},
    "HI_PRED": {"color": "gold",   "linestyle": "--"},
    "TP":      {"color": "red",    "linestyle": "-"},
    "CUSUM":   {"color": "purple", "linestyle": "-"},
    "UCL":     {"color": "orange", "linestyle": "--"},
    "X":       {"color": "blue",   "linestyle": "-"},
    "Y":       {"color": "orange", "linestyle": "-"},
    "Z":       {"color": "green",  "linestyle": "-"},
}

## Import

In [ ]:
import os

import numpy as np
from numpy.fft import rfft, rfftfreq
from scipy.stats import kurtosis, skew

import pandas as pd

import dask.dataframe as dd
from dask.diagnostics.progress import ProgressBar

from matplotlib import pyplot as plt

from tqdm import tqdm

import gc

## Helper Functions

In [ ]:
# ─── Path Helpers ─────────────────────────────────────────────────────────────
def bearing_root(name: str) -> str:
    return f"{ROOT_PROJECT_DATASET}/{name}"

def bearing_full_parquet(name: str) -> str:
    """File mentah hasil akuisisi (bearing_1_full.parquet)"""
    return f"{bearing_root(name)}/{name}_full.parquet"

def export_dataset_root(name: str) -> str:
    return f"{bearing_root(name)}/datasets"

def export_raw_base(name: str) -> str:
    """Dataset raw: time, X, Y, Z rata-rata per detik (sama seperti versi X)"""
    return f"{export_dataset_root(name)}/raw/{name}_raw"

def export_stat_fft_base(name: str) -> str:
    """Dataset fitur statistik time-domain + spectral (FFT)"""
    return f"{export_dataset_root(name)}/stat_fft/{name}_stat_fft"

In [ ]:
# ─── Feature Extraction ───────────────────────────────────────────────────────
def extract_window_features(window: np.ndarray, fs: int = 2560) -> dict:
    """
    Mengekstrak fitur time-domain dan frequency-domain dari 1 window sinyal.

    Parameters
    ----------
    window : np.ndarray, shape (N,)
        Sinyal 1 sumbu dalam 1 window.
    fs : int
        Sampling rate (Hz).

    Returns
    -------
    dict berisi fitur time-domain dan spectral.
    """
    # ── Time-Domain ──────────────────────────────────────────────────────────
    rms          = np.sqrt(np.mean(window ** 2))
    peak         = np.max(np.abs(window))
    peak2peak    = np.max(window) - np.min(window)
    crest_factor = peak / rms if rms > 0 else 0.0
    kurt         = float(kurtosis(window, fisher=True))   # excess kurtosis
    skewness     = float(skew(window))

    # ── Frequency-Domain (via rfft) ──────────────────────────────────────────
    fft_mag  = np.abs(rfft(window))
    freqs    = rfftfreq(len(window), d=1.0 / fs)

    spectral_energy   = np.sum(fft_mag ** 2)
    total_mag         = np.sum(fft_mag)
    spectral_mean     = np.sum(freqs * fft_mag) / total_mag if total_mag > 0 else 0.0
    spectral_variance = np.sum(((freqs - spectral_mean) ** 2) * fft_mag) / total_mag if total_mag > 0 else 0.0

    return {
        "rms":               rms,
        "kurtosis":          kurt,
        "skewness":          skewness,
        "peak2peak":         peak2peak,
        "crest_factor":      crest_factor,
        "spectral_energy":   spectral_energy,
        "spectral_mean":     spectral_mean,
        "spectral_variance": spectral_variance,
    }


def build_feature_dataframe(raw_df: pd.DataFrame, fs: int = 2560) -> pd.DataFrame:
    """
    Memproses seluruh DataFrame raw (time, X, Y, Z) menjadi fitur statistik
    per window 1 detik (fs sampel).

    Returns
    -------
    pd.DataFrame dengan kolom:
        time,
        RMS_X/Y/Z, Kurtosis_X/Y/Z, Skewness_X/Y/Z,
        Peak2Peak_X/Y/Z, CrestFactor_X/Y/Z,
        SpectralEnergy_X/Y/Z, SpectralMean_X/Y/Z, SpectralVariance_X/Y/Z
    """
    n_rows   = len(raw_df)
    n_windows = n_rows // fs

    records = []
    for i in tqdm(range(n_windows), desc="Extracting features", unit="window"):
        start = i * fs
        end   = start + fs
        chunk = raw_df.iloc[start:end]

        t = float(chunk["time"].iloc[0])

        row = {"time": t}
        for axis in ("X", "Y", "Z"):
            feats = extract_window_features(chunk[axis].values, fs=fs)
            row[f"RMS_{axis}"]               = feats["rms"]
            row[f"Kurtosis_{axis}"]           = feats["kurtosis"]
            row[f"Skewness_{axis}"]           = feats["skewness"]
            row[f"Peak2Peak_{axis}"]          = feats["peak2peak"]
            row[f"CrestFactor_{axis}"]        = feats["crest_factor"]
            row[f"SpectralEnergy_{axis}"]     = feats["spectral_energy"]
            row[f"SpectralMean_{axis}"]       = feats["spectral_mean"]
            row[f"SpectralVariance_{axis}"]   = feats["spectral_variance"]

        records.append(row)

    return pd.DataFrame(records).reset_index(drop=True)

In [ ]:
# ─── CUSUM Turning Point Detection ───────────────────────────────────────────
def detect_tcp_cusum(
    series: pd.Series,
    baseline_ratio: float = 0.10,
    k_factor: float = 0.5,
    h_factor: float = 5.0,
) -> tuple[int, np.ndarray, float]:
    """
    Deteksi Turning Point (Tcp) menggunakan Upper CUSUM.

    Parameters
    ----------
    series         : sinyal 1D (misal RMS gabungan X+Y+Z).
    baseline_ratio : proporsi data awal yang diasumsikan sehat.
    k_factor       : pengali sigma untuk drift threshold k.
    h_factor       : pengali sigma untuk decision threshold h.

    Returns
    -------
    (tcp_index, cusum_scores, h_threshold)
    """
    values = series.values.astype(float)
    n      = len(values)

    baseline_n = max(int(n * baseline_ratio), 2)
    mu_0  = values[:baseline_n].mean()
    sigma = values[:baseline_n].std()

    k = k_factor * sigma
    h = h_factor * sigma

    s_plus     = np.zeros(n)
    tcp_index  = n - 1  # default: tidak ada degradasi terdeteksi
    detected   = False

    for t in range(1, n):
        s_plus[t] = max(0.0, s_plus[t - 1] + (values[t] - mu_0 - k))
        if s_plus[t] > h and not detected:
            tcp_index = t
            detected  = True

    print(f"[CUSUM] μ₀={mu_0:.4f}, σ={sigma:.4f}, k={k:.4f}, h={h:.4f}")
    print(f"[CUSUM] Turning Point Index → {tcp_index}")

    return tcp_index, s_plus, h

In [ ]:
# ─── Health Index Labeling ────────────────────────────────────────────────────
def calculate_health_index(t: float, Tcp: float, Tf: float) -> float:
    """HI = 1.0 (sehat) sampai Tcp, kemudian turun linier hingga 0.0 di Tf."""
    return 1.0 - (max(t - Tcp, 0.0) / (Tf - Tcp))


# ─── Visualization ────────────────────────────────────────────────────────────
def plot_features_with_hi(
    df: pd.DataFrame,
    feature_cols: list[str],
    turning_point_time: float,
    title: str,
) -> None:
    """Plot fitur vs Health Index dengan dual y-axis."""
    fig, ax1 = plt.subplots(figsize=(16, 6))

    for col in feature_cols:
        ax1.plot(df["time"], df[col], label=col, alpha=0.8)

    ax1.axvline(turning_point_time, **PLOT_STYLES["TP"], linewidth=2, label="Turning Point")
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Feature Value")
    ax1.tick_params(axis="x", rotation=45)

    ax2 = ax1.twinx()
    ax2.plot(df["time"], df["health_index"], label="Health Index", **PLOT_STYLES["HI"])
    ax2.set_ylabel("Health Index", color=PLOT_STYLES["HI"]["color"])
    ax2.tick_params(axis="y", labelcolor=PLOT_STYLES["HI"]["color"])

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=9)

    plt.title(title)
    plt.tight_layout()
    plt.show()

## A. Load Raw Data

### 1. Baca bearing_1_full.parquet

In [ ]:
INPUT_PARQUET = bearing_full_parquet(MAIN_BEARING)
print(f"Reading: {INPUT_PARQUET}")

_raw_ddf = dd.read_parquet(INPUT_PARQUET)
print(f"Partitions : {_raw_ddf.npartitions}")
_raw_ddf.head(3)

### 2. Compute ke memori

In [ ]:
with ProgressBar():
    RAW_DF = _raw_ddf[["time", "X", "Y", "Z"]].compute().reset_index(drop=True)

BEARING_LIFESPAN_TIME = float(RAW_DF["time"].max())

print(f"Total sampel  : {len(RAW_DF):,}")
print(f"Durasi total  : {BEARING_LIFESPAN_TIME:,.0f} detik  ({BEARING_LIFESPAN_TIME/3600:.1f} jam)")
print(f"Total window  : {len(RAW_DF) // SAMPLING_RATE:,} detik")

del _raw_ddf
gc.collect()

## B. Build Raw Dataset
Group raw data per detik (2560 sampel → 1 baris). Sama persis dengan pendekatan si X.

In [ ]:
# Group by detik: time=first, X/Y/Z=mean  (identik dengan versi X)
RAW_GROUPED = (
    RAW_DF
    .assign(time_group=(RAW_DF["time"] // 1).astype("int32"))
    .groupby("time_group", sort=True)
    .agg(time=("time", "first"), X=("X", "mean"), Y=("Y", "mean"), Z=("Z", "mean"))
    .reset_index(drop=True)
)

print(f"Shape RAW_GROUPED : {RAW_GROUPED.shape}")
print(f"Kolom             : {list(RAW_GROUPED.columns)}")
RAW_GROUPED.head(3)

In [ ]:
# RAW_DF masih dibutuhkan di Section C untuk feature extraction
# (akan dihapus setelah feature extraction selesai)

## C. Feature Extraction
Per window 1 detik (2560 sampel). Menghasilkan fitur statistik:
- **Time-domain**: RMS, Kurtosis, Skewness, Peak2Peak, CrestFactor
- **Freq-domain**: SpectralEnergy, SpectralMean, SpectralVariance

In [ ]:
FEAT_DF = build_feature_dataframe(RAW_DF, fs=SAMPLING_RATE)

print(f"Shape hasil ekstraksi : {FEAT_DF.shape}")
print(f"Kolom                 : {list(FEAT_DF.columns)}")
FEAT_DF.head(3)

In [ ]:
# Bebaskan memori raw data setelah feature extraction selesai
del RAW_DF
gc.collect()

## D. CUSUM — Auto Turning Point Detection

### 1. Deteksi Turning Point

In [ ]:
# Gunakan rata-rata RMS ketiga sumbu sebagai sinyal referensi
cusum_signal = (FEAT_DF["RMS_X"] + FEAT_DF["RMS_Y"] + FEAT_DF["RMS_Z"]) / 3

tcp_idx, cusum_scores, cusum_threshold = detect_tcp_cusum(
    series=cusum_signal,
    baseline_ratio=CUSUM_BASELINE_RATIO,
    k_factor=CUSUM_K_FACTOR,
    h_factor=CUSUM_H_FACTOR,
)

TURNING_POINT_TIME = float(FEAT_DF["time"].iloc[tcp_idx])
print(f"\nTURNING_POINT_TIME = {TURNING_POINT_TIME:,.0f} detik  ({TURNING_POINT_TIME/3600:.2f} jam)")

### 2. Visualisasi CUSUM Score

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

# ── Panel atas: Mean RMS signal ──────────────────────────────────────────────
axes[0].plot(FEAT_DF["time"], cusum_signal, label="Mean RMS (X+Y+Z)/3", **PLOT_STYLES["X"], alpha=0.8)
axes[0].axvline(TURNING_POINT_TIME, **PLOT_STYLES["TP"], linewidth=2, label=f"Turning Point ({TURNING_POINT_TIME:,.0f} s)")
axes[0].set_ylabel("Mean RMS")
axes[0].set_title("Sinyal RMS — Deteksi Degradasi Bearing")
axes[0].legend(loc="upper left")
axes[0].grid(True, alpha=0.3)

# ── Panel bawah: CUSUM Score ─────────────────────────────────────────────────
axes[1].plot(FEAT_DF["time"], cusum_scores, label="CUSUM Score (S+)", **PLOT_STYLES["CUSUM"])
axes[1].axhline(cusum_threshold, **PLOT_STYLES["UCL"], linewidth=1.5, label=f"Threshold h = {cusum_threshold:.4f}")
axes[1].axvline(TURNING_POINT_TIME, **PLOT_STYLES["TP"], linewidth=2, label=f"Turning Point ({TURNING_POINT_TIME:,.0f} s)")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("CUSUM Score")
axes[1].set_title("CUSUM Score — S+ melampaui threshold → Turning Point")
axes[1].legend(loc="upper left")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## E. Labeling Health Index

In [ ]:
# ── Terapkan ke FEAT_DF (Stat+FFT) ───────────────────────────────────────────
FEAT_DF["health_index"] = FEAT_DF["time"].apply(
    lambda t: calculate_health_index(t, TURNING_POINT_TIME, BEARING_LIFESPAN_TIME)
)

# ── Terapkan ke RAW_GROUPED ───────────────────────────────────────────────────
RAW_GROUPED["health_index"] = RAW_GROUPED["time"].apply(
    lambda t: calculate_health_index(t, TURNING_POINT_TIME, BEARING_LIFESPAN_TIME)
)

print(f"HI range (FEAT_DF)    : [{FEAT_DF['health_index'].min():.4f}, {FEAT_DF['health_index'].max():.4f}]")
print(f"HI range (RAW_GROUPED): [{RAW_GROUPED['health_index'].min():.4f}, {RAW_GROUPED['health_index'].max():.4f}]")

## F. Data Smoothing

### 1. Rolling Mean

In [ ]:
FEATURE_COLS = [c for c in FEAT_DF.columns if c not in ("time", "health_index")]
TIME_DOMAIN_COLS = [c for c in FEATURE_COLS if not c.startswith("Spectral")]
SPECTRAL_COLS    = [c for c in FEATURE_COLS if c.startswith("Spectral")]

print(f"Time-domain features  ({len(TIME_DOMAIN_COLS)}): {TIME_DOMAIN_COLS}")
print(f"Spectral features     ({len(SPECTRAL_COLS)}): {SPECTRAL_COLS}")

# ── Smooth Stat+FFT Dataset ───────────────────────────────────────────────────
FEAT_DF_SMOOTH = FEAT_DF.copy()
FEAT_DF_SMOOTH[FEATURE_COLS] = (
    FEAT_DF[FEATURE_COLS]
    .rolling(window=SMOOTHING_WINDOW, min_periods=1)
    .mean()
)

# ── Smooth Raw Dataset ────────────────────────────────────────────────────────
RAW_GROUPED_SMOOTH = RAW_GROUPED.copy()
RAW_GROUPED_SMOOTH[["X", "Y", "Z"]] = (
    RAW_GROUPED[["X", "Y", "Z"]]
    .rolling(window=SMOOTHING_WINDOW, min_periods=1)
    .mean()
)

## G. Data Cleaning

In [ ]:
FEAT_DF_SMOOTH    = FEAT_DF_SMOOTH.dropna().reset_index(drop=True)
RAW_GROUPED_SMOOTH = RAW_GROUPED_SMOOTH.dropna().reset_index(drop=True)

print(f"Baris FEAT (setelah cleaning): {len(FEAT_DF_SMOOTH):,}")
print(f"Baris RAW  (setelah cleaning): {len(RAW_GROUPED_SMOOTH):,}")

## H. Down Sampling & Export

### 1. Down Sampling

In [ ]:
# ── Stat+FFT Dataset ──────────────────────────────────────────────────────────
STAT_FFT_TRAIN = FEAT_DF_SMOOTH.iloc[::DOWN_SAMPLING_FACTOR].reset_index(drop=True)
STAT_FFT_TEST  = FEAT_DF_SMOOTH.iloc[TESTING_INITIAL_POINT::DOWN_SAMPLING_FACTOR].reset_index(drop=True)

# ── Raw Dataset ───────────────────────────────────────────────────────────────
RAW_TRAIN = RAW_GROUPED_SMOOTH.iloc[::DOWN_SAMPLING_FACTOR].reset_index(drop=True)
RAW_TEST  = RAW_GROUPED_SMOOTH.iloc[TESTING_INITIAL_POINT::DOWN_SAMPLING_FACTOR].reset_index(drop=True)

print(f"STAT+FFT → Train: {len(STAT_FFT_TRAIN):,} | Test: {len(STAT_FFT_TEST):,}")
print(f"RAW      → Train: {len(RAW_TRAIN):,}     | Test: {len(RAW_TEST):,}")

### 2. Export — Raw (time, X, Y, Z)

In [ ]:
RAW_EXPORT_COLS = ["time", "health_index", "X", "Y", "Z"]

os.makedirs(os.path.dirname(export_raw_base(MAIN_BEARING)), exist_ok=True)

train_path_raw = f"{export_raw_base(MAIN_BEARING)}_train.parquet"
test_path_raw  = f"{export_raw_base(MAIN_BEARING)}_test.parquet"

RAW_TRAIN[RAW_EXPORT_COLS].to_parquet(train_path_raw, engine="pyarrow", compression="snappy")
RAW_TEST[RAW_EXPORT_COLS].to_parquet(test_path_raw,   engine="pyarrow", compression="snappy")

print(f"[RAW] Train → {train_path_raw}")
print(f"[RAW] Test  → {test_path_raw}")

plot_features_with_hi(
    df=RAW_TRAIN[RAW_EXPORT_COLS],
    feature_cols=["X", "Y", "Z"],
    turning_point_time=TURNING_POINT_TIME,
    title="Training Dataset (Raw) — X, Y, Z Rata-Rata per Detik",
)

### 3. Export — Stat + FFT (Time-Domain + Spectral)

In [ ]:
STAT_FFT_COLS = ["time", "health_index"] + FEATURE_COLS

os.makedirs(os.path.dirname(export_stat_fft_base(MAIN_BEARING)), exist_ok=True)

train_path_fft = f"{export_stat_fft_base(MAIN_BEARING)}_train.parquet"
test_path_fft  = f"{export_stat_fft_base(MAIN_BEARING)}_test.parquet"

STAT_FFT_TRAIN[STAT_FFT_COLS].to_parquet(train_path_fft, engine="pyarrow", compression="snappy")
STAT_FFT_TEST[STAT_FFT_COLS].to_parquet(test_path_fft,   engine="pyarrow", compression="snappy")

print(f"[STAT+FFT] Train → {train_path_fft}")
print(f"[STAT+FFT] Test  → {test_path_fft}")

plot_features_with_hi(
    df=STAT_FFT_TRAIN[STAT_FFT_COLS],
    feature_cols=["SpectralEnergy_X", "SpectralEnergy_Y", "SpectralEnergy_Z"],
    turning_point_time=TURNING_POINT_TIME,
    title="Training Dataset (Stat+FFT) — Spectral Energy per Axis",
)

## Ringkasan Output

| Dataset | Kolom | File |
|---------|-------|------|
| `raw_train` | time, HI, X, Y, Z (rata-rata per detik) | `…/raw/bearing_1_raw_train.parquet` |
| `raw_test`  | sama | `…/raw/bearing_1_raw_test.parquet` |
| `stat_fft_train` | time, HI, RMS×3, Kurtosis×3, Skewness×3, Peak2Peak×3, CrestFactor×3, SpectralEnergy×3, SpectralMean×3, SpectralVariance×3 | `…/stat_fft/bearing_1_stat_fft_train.parquet` |
| `stat_fft_test`  | sama | `…/stat_fft/bearing_1_stat_fft_test.parquet` |